# Slope Investigation

Reusable diagnostics for inspecting hydraulic-context slopes across one or more examples.

## Imports

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "RAPID" / "src" / "rapid_tools").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing RAPID/src/rapid_tools")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
if str(RAPID_SRC) not in sys.path:
    sys.path.append(str(RAPID_SRC))

if "rapid_tools" in sys.modules:
    importlib.reload(sys.modules["rapid_tools"])

from rapid_tools import (
    build_slope_link_frame,
    build_low_k_parent_summary,
    build_slope_node_frame,
    identify_slope_seed_links,
    list_slope_states,
    plot_actual_sword_vs_state_vs_monotone,
    plot_raw_vs_monotone_profile,
    plot_slope_plan_view,
    plot_slope_profiles,
    plot_slope_ratio_scatter,
    plot_sword_vs_monotone_profile,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Configuration

In [52]:
EXAMPLE_IDS = ["02"]
MODE = "indep"  # or "compound"
STATUS = "failed"  # use None to load all states from the prep registry
INCLUDE_BASE = True
LIMIT_PER_EXAMPLE = 4
ANCHORED_SUFFIX = "__anchored"

# Optional override. Use either None, a shared list like ["S000_base", "S001_unit_2"],
# or a dict like {"10": ["S000_base", "S001_unit_2"], "56": ["S000_base", "S009_unit_1"]}
STATE_IDS = None

# Optional focus state for raw-vs-monotone inspection.
# Leave as None to facet over all loaded states.
PROFILE_FOCUS_EXAMPLE_ID = "02"
PROFILE_FOCUS_STATE_ID = None

LOW_K_STATE_ID = "S000_base"
LOW_K_TOP_N = 20

## Load Diagnostics

In [53]:
selection = list_slope_states(
    EXAMPLE_IDS,
    mode=MODE,
    status=STATUS,
    include_base=INCLUDE_BASE,
    limit_per_example=LIMIT_PER_EXAMPLE,
    anchored_suffix=ANCHORED_SUFFIX,
    state_ids=STATE_IDS,
)

node_df = build_slope_node_frame(selection)
link_df = build_slope_link_frame(selection)
seed_df = identify_slope_seed_links(link_df, min_ratio=3.0)

print(f"Loaded {len(selection)} states, {len(node_df)} node rows, and {len(link_df)} link rows.")
display(selection[["example_id", "state_id", "state_role", "prep_status", "prep_error"]])
display(seed_df[[
    c for c in [
        "example_id",
        "state_id",
        "id_link",
        "dominant_parent_link_id",
        "lineage_type",
        "raw_slope",
        "slope_section_ref",
        "slope_section_ratio",
        "slope_used",
    ] if c in seed_df.columns
]])

Loaded 1 states, 60 node rows, and 88 link rows.


,example_id,state_id,state_role,prep_status,prep_error
0,02,S000_base,base,prepared,None


,example_id,state_id,id_link,dominant_parent_link_id,lineage_type,raw_slope,slope_section_ref,slope_section_ratio,slope_used
18,02,S000_base,76,76,base_identity_1to1,0.001311,0.000421,3.113745,0.001311


## Along-Distance WSE Profiles

In [54]:
fig = plot_actual_sword_vs_state_vs_monotone(
    node_df,
    example_id=PROFILE_FOCUS_EXAMPLE_ID,
    state_id=PROFILE_FOCUS_STATE_ID,
)
fig.show()

In [55]:
fig = plot_sword_vs_monotone_profile(
    node_df,
    example_id=PROFILE_FOCUS_EXAMPLE_ID,
    state_id=PROFILE_FOCUS_STATE_ID,
)
fig.show()


In [45]:
fig = plot_raw_vs_monotone_profile(
    node_df,
    example_id=PROFILE_FOCUS_EXAMPLE_ID,
    state_id=PROFILE_FOCUS_STATE_ID,
)
fig.show()

In [46]:
fig = plot_slope_profiles(node_df)
fig.show()

## Plan-View Link Diagnostics

In [47]:
fig = plot_slope_plan_view(link_df, color_column="slope_used")
fig.show()

## Raw-Slope Versus Section-Ratio Diagnostics

In [48]:
fig = plot_slope_ratio_scatter(link_df)
fig.show()

## Highest-Ratio Links

In [10]:
display(
    link_df[[
        c for c in [
            "example_id",
            "state_id",
            "id_link",
            "dominant_parent_link_id",
            "lineage_type",
            "slope_source_method",
            "raw_slope",
            "slope_section_ref",
            "slope_section_ratio",
            "slope_used",
            "slope_neighbor_source_link_id",
        ] if c in link_df.columns
    ]]
    .sort_values(["example_id", "state_id", "slope_section_ratio"], ascending=[True, True, False])
    .head(50)
)

,example_id,state_id,id_link,dominant_parent_link_id,lineage_type,slope_source_method,raw_slope,slope_section_ref,slope_section_ratio,slope_used,slope_neighbor_source_link_id
38,21,S000_base,21,21,base_identity_1to1,raw_wse,0.001031,0.000238,4.325883,0.001031,NaN
41,21,S000_base,23,23,base_identity_1to1,raw_wse,0.000987,0.000238,4.140203,0.000987,NaN
46,21,S000_base,38,38,base_identity_1to1,raw_wse,0.000888,0.000238,3.726222,0.000888,NaN
65,21,S000_base,42,42,base_identity_1to1,raw_wse,0.000752,0.000238,3.155648,0.000752,NaN
36,21,S000_base,18,18,base_identity_1to1,raw_wse,0.000731,0.000238,3.064575,0.000731,NaN
35,21,S000_base,17,17,base_identity_1to1,compound_bubble_profile,0.000708,0.000238,2.970267,0.000323,NaN
34,21,S000_base,16,16,base_identity_1to1,compound_bubble_profile,0.000683,0.000238,2.866530,0.000476,NaN
68,21,S000_base,47,47,base_identity_1to1,raw_wse,0.000596,0.000238,2.499850,0.000596,NaN
37,21,S000_base,19,19,base_identity_1to1,raw_wse,0.000587,0.000238,2.464123,0.000587,NaN
20,21,S000_base,0,0,base_identity_1to1,raw_wse,0.000566,0.000238,2.375420,0.000566,NaN


## Lowest-K Parent Links

In [29]:
low_k_df = build_low_k_parent_summary(
    selection,
    state_id=LOW_K_STATE_ID,
    top_n=200,
)


In [37]:

# display(
#     low_k_df[[
#         c for c in [
#             "example_id",
#             "state_id",
#             "parent_link_id",
#             "id_us_node",
#             "id_ds_node",
#             "link_length_m",
#             "wid_adj_wet",
#             "slope_used",
#             "min_rapid_k",
#             "median_rapid_k",
#             "max_rapid_celerity_mps",
#             "max_subreach_width_m",
#             "n_subreaches",
#             "parent_centroid_x",
#             "parent_centroid_y",
#         ] if c in low_k_df.columns
#     ]]
# )
low_k_df["dt_max_from_parent_min_k"] = 2 * low_k_df["min_rapid_k"] * 0.9
low_k_df["dt_min_from_parent_max_k"] = 2 * low_k_df["max_rapid_k"] * 0.1

low_k_df[[
    c for c in [
        "state_id",
        "slope_used",
        'wid_adj_wet',
        "min_rapid_k",
        "max_rapid_k",
        "dt_min",
        "dt_max",
        'dt_max_from_parent_min_k',
        'dt_min_from_parent_max_k'
    ] if c in low_k_df.columns
]].sort_values("max_rapid_k", ascending = False).head(20)


,state_id,slope_used,wid_adj_wet,min_rapid_k,max_rapid_k,dt_min,dt_max,dt_max_from_parent_min_k,dt_min_from_parent_max_k
74,S000_base,0.000020,515.952948,3300.382615,4100.972477,820.194495,643.161205,5940.688707,820.194495
78,S000_base,0.000005,793.790352,4074.261260,4074.261260,820.194495,643.161205,7333.670267,814.852252
52,S000_base,0.000085,658.672131,1383.958457,4042.904772,820.194495,643.161205,2491.125222,808.580954
75,S000_base,0.000016,318.337841,3964.382662,4033.583060,820.194495,643.161205,7135.888791,806.716612
71,S000_base,0.000056,165.293169,2394.793181,4000.000000,820.194495,643.161205,4310.627725,800.000000
55,S000_base,0.000241,33.309239,1421.435440,4000.000000,820.194495,643.161205,2558.583792,800.000000
69,S000_base,0.000053,231.118694,1956.039490,4000.000000,820.194495,643.161205,3520.871082,800.000000
73,S000_base,0.000046,231.498274,3039.860170,4000.000000,820.194495,643.161205,5471.748307,800.000000
76,S000_base,0.000003,228.009477,4000.000000,4000.000000,820.194495,643.161205,7200.000000,800.000000
77,S000_base,0.000005,1148.911946,4000.000000,4000.000000,820.194495,643.161205,7200.000000,800.000000


In [ ]:
(2*low_k_df['min_rapid_k'])*(1-0.1)
(2*low_k_df['max_rapid_k'])*0.1

0      71.462356
1     110.014538
2     200.393729
3     106.965421
4     110.738503
5      93.269456
6     109.464521
7     125.432227
8     113.045430
9     202.152504
10    195.142073
11    196.107181
12    335.112098
13    251.542619
14    249.678146
15    232.639489
16    179.309452
17    373.218931
18    300.936171
19    671.684390
Name: max_rapid_k, dtype: float64

In [25]:
low_k_df

,parent_link_id,min_rapid_k,median_rapid_k,max_rapid_k,max_rapid_celerity_mps,min_rapid_celerity_mps,min_subreach_width_m,max_subreach_width_m,n_subreaches,id_us_node,id_ds_node,link_length_m,len,wid_adj_wet,slope_used,parent_centroid_x,parent_centroid_y,example_id,mode,state_id,state_label,dt_min,dt_max
0,38,357.311781,357.311781,357.311781,0.559735,0.559735,1139.033276,1139.033276,1,24,23,117.241076,117.241076,1141.199914,0.000888,175476.743885,2.001103e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
1,42,395.404974,442.912342,550.072690,0.505811,0.363588,675.479751,1108.355977,30,25,28,5925.637717,5925.637717,951.905499,0.000752,162536.828844,1.999945e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
2,0,412.803454,496.709970,1001.968645,0.484492,0.199607,339.994860,1285.696804,64,1,0,12724.939547,12724.939547,907.403204,0.000566,194100.845005,2.016111e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
3,23,418.546782,493.940427,534.827104,0.477844,0.373953,574.741513,830.188415,13,18,14,2560.102398,2560.102398,631.762691,0.000987,182630.062888,2.007202e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
4,44,429.501364,459.498494,553.692517,0.466642,0.361976,873.411757,1278.424820,16,33,27,3206.770712,3206.770712,1105.753142,0.000529,173593.010827,1.998614e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
5,21,442.721105,452.937019,466.347278,0.451752,0.428865,683.028233,738.426862,4,14,12,751.046818,751.046818,714.431652,0.001031,183682.139610,2.008284e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
6,47,445.381129,520.605591,547.322607,0.455209,0.370424,827.233213,1126.925809,7,32,29,1419.191675,1419.191675,858.429775,0.000596,157694.153413,1.998318e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
7,30,450.895702,523.522030,627.161137,0.445797,0.320504,779.746024,1279.109526,39,23,19,7839.305338,7839.305338,980.488660,0.000483,177874.580616,2.003751e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
8,18,514.520869,534.527398,565.227152,0.389482,0.354541,664.870102,765.538147,15,12,11,3005.947099,3005.947099,714.776886,0.000731,185015.055240,2.009343e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
9,54,616.075330,899.290402,1010.762521,0.324636,0.197870,464.059243,975.206069,14,34,36,2731.963115,2731.963115,522.588741,0.000368,169482.380804,1.996907e+06,21,indep,S000_base,21 · S000_base,820.194495,643.161205
